In [132]:
import scenic
import tempfile
import pathlib

min = -300
max = 300
step = 100
closeness = 20

trace_count = 1000

shortLen = 20
longLen = 40
testTracesShort = []
testTracesLong = []


from scenic.simulators.newtonian import NewtonianSimulator


scenario = scenic.scenarioFromFile('Scenic/examples/driving/badlyParkedCarPullingIn.scenic',
                                   model='scenic.simulators.newtonian.driving_model',
                                   mode2D=True)


for n in range(trace_count):
    
    scene, _ = scenario.generate()

    data = scenario.sceneToBytes(scene)
    with open(pathlib.Path(tempfile.gettempdir()) / 'test.scene', 'wb') as f:
        f.write(data)


#SHORT TRACES
    
    simulator = NewtonianSimulator()
    simulation = simulator.simulate(scene, maxSteps=shortLen-1)
    if simulation: 
        result = simulation.result
        traces = []
        for i, state in enumerate(result.trajectory):
            egoPos, parkedCarPos = state
            states = []

            for x in range(min,max,step): 
                for y in range(min,max,step): 
                    if (x < egoPos[0] <= x+step) and (y < egoPos[1] <=y+step):
                        states.append(((x, x+step),(y, y+step)))
                 
            
            for z in range(min,max,step): 
                for v in range(min,max,step): 
                    if (z < parkedCarPos[0] <= z+step) and (v < parkedCarPos[1] <=v+step):
                        states.append(((z, z+step),(v, v+step)))
                     
                        
            if abs((egoPos[0]) - (parkedCarPos[0])) < closeness and abs((egoPos[1]) - (parkedCarPos[1])) < closeness: 
                states.append('close')    
            else: 
                states.append('far')

            if abs(egoPos[0] - parkedCarPos[0]) < 4.5 and abs(egoPos[1] - parkedCarPos[1]) < 2:
                states.append('collision')
            else:
                states.append('no_collision')

            
            
            traces.append(states)
        testTracesShort.append(traces)

     
#LONG TRACES - EXTENSIONS OF SHORT TRACES 
    
    with open(pathlib.Path(tempfile.gettempdir()) / 'test.scene', 'rb') as f:
        data = f.read()
        scene = scenario.sceneFromBytes(data)

           
    simulator = NewtonianSimulator()
    simulation = simulator.simulate(scene, maxSteps=longLen-1)
    if simulation:  
        result = simulation.result
        traces = []
        for i, state in enumerate(result.trajectory):
            egoPos, parkedCarPos = state
            states = []

            for x in range(min,max,step): 
                for y in range(min,max,step): 
                    if (x < egoPos[0] <= x+step) and (y < egoPos[1] <=y+step):
                        states.append(((x, x+step),(y, y+step)))
                    
            
            for z in range(min,max,step): 
                for v in range(min,max,step): 
                    if (z < parkedCarPos[0] <= z+step) and (v < parkedCarPos[1] <=v+step):
                        states.append(((z, z+step),(v, v+step)))
                        
                        
            if abs((egoPos[0]) - (parkedCarPos[0])) < closeness and abs((egoPos[1]) - (parkedCarPos[1])) < closeness: 
                states.append('close')    
            else: 
                states.append('far')

            if abs(egoPos[0] - parkedCarPos[0]) < 4.5 and abs(egoPos[1] - parkedCarPos[1]) < 2:
                states.append('collision')     
            else:
                states.append('no_collision')

            
            
            traces.append(states) 
        testTracesLong.append(traces)


In [133]:
testTraces = {}

for n in range(trace_count):
    testTraces[n] = 0 
 
for n in range(trace_count):
    testTraces[n] = testTracesShort[n]


for n in range(trace_count):
    for s in testTracesShort[n]: 
        if s[3] == 'collision': 
            testTraces[n] = 'error' 
            

    




In [134]:
testTraces_willCrashAfter20Steps  = {}

for n in range(trace_count):
  testTraces_willCrashAfter20Steps[n] = 0   

for n in range(trace_count):
    for t in testTracesLong[n][20:]:
        if t[3] == 'collision':
            testTraces_willCrashAfter20Steps[n] = 1  


for n in range(trace_count):
    if testTraces[n] == 'error': 
        testTraces_willCrashAfter20Steps[n] = 'error'
    

In [146]:
import numpy as np 
np.save('testTraces', testTraces) 

read_testTraces = np.load('testTraces.npy',allow_pickle='TRUE').item()
 

In [148]:
import numpy as np 
np.save('testTraces_willCrashAfter20Steps', testTraces_willCrashAfter20Steps) 

read_testTraces_willCrashAfter20Steps = np.load('testTraces_willCrashAfter20Steps.npy',allow_pickle='TRUE').item()